# run — COME / CoTTA vs BMIA (tab:come_cotta)
**Professor B**  
Goal: Compare COME and CoTTA baselines against BMIA on CIFAR-100-C.  
Output: accuracy per method × corruption → `V13_COME_COTTA.json`

**Kaggle setup**: Add Data → `hendrycks/cifar100c` | GPU T4 x2 | Internet: ON

In [ ]:
import os, json, copy, gc, time, random
import numpy as np
import torch, torch.nn as nn, torch.optim as optim
import torchvision, torchvision.transforms as T
from torch.utils.data import DataLoader

torch.manual_seed(42); np.random.seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
N_GPUS = torch.cuda.device_count()
print(f'Device: {device}  GPUs: {N_GPUS}')

C100C_ROOT = None
for root, dirs, files in os.walk('/kaggle/input/'):
    if 'gaussian_noise.npy' in files and 'labels.npy' in files:
        C100C_ROOT = root; break
assert C100C_ROOT, 'Add dataset: hendrycks/cifar100c'
print(f'CIFAR-100-C: {C100C_ROOT}')

NUM_CLASSES = 100
TTA_BATCH   = 64
EVAL_BATCH  = 512
LR          = 0.005
TAU         = 0.10
SEVERITY    = 5
SEV_OFFSET  = (SEVERITY-1)*10000
CORRUPTIONS = ['gaussian_noise','defocus_blur','snow','contrast','jpeg_compression']
METHODS     = ['tent','sar','eata','cotta','come','bmia_fixed','bmia_adaptive','bmia_adaptive_v3']
C_MEAN = torch.tensor([0.5071,0.4867,0.4408]).view(1,3,1,1)
C_STD  = torch.tensor([0.2675,0.2565,0.2761]).view(1,3,1,1)

def load_c100c(c):
    data  =np.load(f'{C100C_ROOT}/{c}.npy')[SEV_OFFSET:SEV_OFFSET+10000]
    lbls  =np.load(f'{C100C_ROOT}/labels.npy')
    X=(torch.tensor(data).permute(0,3,1,2).float()/255.0-C_MEAN)/C_STD
    return X.to(device), torch.tensor(lbls,dtype=torch.long).to(device)

print('Setup OK.')

In [ ]:
# ── Train ResNet-18 ──────────────────────────────────────────────
ds = torchvision.datasets.CIFAR100('/tmp',train=True,download=True,transform=T.Compose([
    T.RandomCrop(32,4),T.RandomHorizontalFlip(),
    T.ToTensor(),T.Normalize([0.5071,0.4867,0.4408],[0.2675,0.2565,0.2761])]))
dl = DataLoader(ds,batch_size=128,shuffle=True,num_workers=4,pin_memory=True)

model=torchvision.models.resnet18(weights=None)
model.conv1=nn.Conv2d(3,64,3,stride=1,padding=1,bias=False)
model.maxpool=nn.Identity()
model.fc=nn.Linear(512,NUM_CLASSES)
model=model.to(device)
dp=nn.DataParallel(model) if N_GPUS>1 else model
opt_tr=optim.SGD(model.parameters(),lr=0.1,momentum=0.9,weight_decay=5e-4)
sch=optim.lr_scheduler.CosineAnnealingLR(opt_tr,T_max=30)
crit=nn.CrossEntropyLoss()
dp.train(); t0=time.time()
for ep in range(30):
    for x,y in dl:
        x,y=x.to(device),y.to(device)
        opt_tr.zero_grad(); crit(dp(x),y).backward(); opt_tr.step()
    sch.step()
    if (ep+1)%10==0: print(f'  ep={ep+1}/30  {(time.time()-t0)/60:.1f}min')
model.eval(); SOURCE_STATE=copy.deepcopy(model.state_dict())
del dp,dl,ds; gc.collect(); torch.cuda.empty_cache()
print('Training done.')

In [ ]:
# ── Helpers ──────────────────────────────────────────────────────
def freeze_bn(mdl):
    ids={id(p) for m in mdl.modules() if isinstance(m,nn.BatchNorm2d) for p in m.parameters()}
    for p in mdl.parameters(): p.requires_grad_(id(p) in ids)

def get_acc(mdl,X,y):
    mdl.eval(); preds=[]
    with torch.no_grad():
        for i in range(0,X.size(0),EVAL_BATCH):
            preds.append(mdl(X[i:i+EVAL_BATCH]).argmax(1))
    preds=torch.cat(preds)
    return round((preds==y[:len(preds)]).float().mean().item(),4)

def run_method(X, y, method):
    model.load_state_dict(copy.deepcopy(SOURCE_STATE))
    model.train(); freeze_bn(model)
    params=[p for p in model.parameters() if p.requires_grad]
    opt=optim.SGD(params,lr=LR,momentum=0.9)
    init={pn:p.data.clone() for pn,p in model.named_parameters() if p.requires_grad}
    lam=({'bmia_fixed':1.0,'bmia_adaptive':0.5,'bmia_adaptive_v3':0.1}).get(method,0.5)
    is_fixed=(method=='bmia_fixed'); use_v3=(method=='bmia_adaptive_v3')
    step=TTA_BATCH*2 if use_v3 else TTA_BATCH
    fisher=None

    if method=='eata':
        fisher={pn:torch.zeros_like(p) for pn,p in model.named_parameters() if p.requires_grad}
        for j in range(0,min(128,X.size(0)),TTA_BATCH):
            xb=X[j:j+TTA_BATCH]
            if xb.size(0)<2: continue
            model.zero_grad(); pr=torch.softmax(model(xb),1)
            (-(pr*torch.log(pr+1e-8)).sum(1).mean()).backward()
            for pn,p in model.named_parameters():
                if pn in fisher and p.grad is not None: fisher[pn]+=p.grad.data**2*xb.size(0)
        for pn in fisher: fisher[pn]/=128
        model.load_state_dict(copy.deepcopy(SOURCE_STATE)); model.train(); freeze_bn(model)
        params=[p for p in model.parameters() if p.requires_grad]
        opt=optim.SGD(params,lr=LR,momentum=0.9)

    # CoTTA teacher model (EMA)
    ema=None
    if method=='cotta':
        ema=copy.deepcopy(model); ema.eval()
        for p in ema.parameters(): p.requires_grad_(False)

    for i in range(0,X.size(0),step):
        if use_v3:
            chunks=[X[i+s*TTA_BATCH:i+(s+1)*TTA_BATCH] for s in range(2)]
            chunks=[c for c in chunks if c.size(0)>=4]
            if not chunks: continue
            xb=torch.cat(chunks)
        else:
            xb=X[i:i+TTA_BATCH]
            if xb.size(0)<4: continue

        opt.zero_grad()
        pr=torch.softmax(model(xb),1)
        ent=-(pr*torch.log(pr+1e-8)).sum(1)

        if method=='tent':
            loss=ent.mean()
        elif method=='sar':
            mask=ent<0.4*np.log(NUM_CLASSES)
            if mask.sum()<2: continue
            loss=ent[mask].mean()
        elif method=='eata':
            mask=ent<0.4*np.log(NUM_CLASSES)
            if mask.sum()<2: continue
            fl=sum((fisher[pn]*(p-init[pn])**2).sum() for pn,p in model.named_parameters() if pn in fisher)
            loss=ent[mask].mean()+2000*fl
        elif method=='cotta':
            # EMA teacher pseudo-labels
            with torch.no_grad():
                p_teacher=torch.softmax(ema(xb),1)
            # Augmented prediction
            xb_flip=torch.flip(xb,[3])
            p_aug=torch.softmax(model(xb_flip),1)
            consist=((p_aug-p_teacher.detach())**2).sum(1)
            loss=ent.mean()+0.5*consist.mean()
            # EMA update
            with torch.no_grad():
                for ep,sp in zip(ema.parameters(),model.parameters()):
                    ep.data.mul_(0.999).add_(sp.data,alpha=0.001)
            # Stochastic restore 1%
            with torch.no_grad():
                for pn,p in model.named_parameters():
                    if p.requires_grad and random.random()<0.01:
                        p.data.copy_(SOURCE_STATE[pn])
        elif method=='come':
            # COME-simplified: entropy + marginal diversity + inter-class contrast
            mg=pr.mean(0); hy=-(mg*torch.log(mg+1e-8)).sum()
            # contrast: high-confidence vs low-confidence mean predictions
            half=len(ent)//2
            idx=ent.argsort()
            p_conf=pr[idx[:half]].mean(0); p_unconf=pr[idx[half:]].mean(0)
            contrast=((p_conf-p_unconf)**2).sum()
            loss=ent.mean()-0.5*hy+0.1*contrast
        else:  # bmia variants
            mg=pr.mean(0); hy=-(mg*torch.log(mg+1e-8)).sum()
            anc=sum(((p-init[pn])**2).sum() for pn,p in model.named_parameters() if pn in init)
            loss=ent.mean()-hy+lam*anc

        loss.backward(); opt.step()

        if not is_fixed and method in('bmia_adaptive','bmia_adaptive_v3'):
            with torch.no_grad():
                dom=(torch.bincount(pr.argmax(1),minlength=NUM_CLASSES).float().max()/xb.size(0)).item()
            lam=min(10.,lam*1.1) if dom>TAU else max(0.01,lam*0.95)

    return get_acc(model,X,y)

print('All methods ready.')

In [ ]:
# ── Run ──────────────────────────────────────────────────────────
results=[]; t0=time.time()
print(f'  {"Corruption":<22} {"Method":<22} Acc')
for corr in CORRUPTIONS:
    X,y=load_c100c(corr)
    for m in METHODS:
        acc=run_method(X,y,m)
        results.append({'corruption':corr,'method':m,'acc':acc,'severity':SEVERITY})
        print(f'  {corr:<22} {m:<22} {acc*100:.1f}%')
        gc.collect(); torch.cuda.empty_cache()
    del X,y; gc.collect(); torch.cuda.empty_cache()
print(f'Done {(time.time()-t0)/60:.1f}min')

print('\nAverage per method:')
for m in METHODS:
    avg=np.mean([r['acc'] for r in results if r['method']==m])*100
    print(f'  {m:<22} {avg:.1f}%')

with open('V13_COME_COTTA.json','w') as f:
    json.dump({'experiment':'V13_COME_COTTA','lr':LR,'severity':SEVERITY,
               'corruptions':CORRUPTIONS,'results':results},f,indent=2)
print('Saved → V13_COME_COTTA.json')
with open('V13_COME_COTTA.json') as f: print(f.read())